In [ ]:
# CELL 1: IMPORTS AND SETUP
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

print("✓ All imports successful")

In [ ]:
# CELL 2: LOAD DATA (HOPV ONLY)
# Load HOPV dataset only (this is what got 0.90)
df_hopv = pd.read_csv('../../../data/002_processed_data/xy_df_hopv.csv')
df_hopv = df_hopv.dropna(subset=['stability'])

df_kaggle = pd.read_csv('../../../data/003_kaggle_competition_data/25261_regression_sample_submission.csv')

print(f"HOPV dataset: {len(df_hopv)} samples")
print(f"Kaggle test: {len(df_kaggle)} samples")
print(f"\nClass distribution:")
print(df_hopv['stability'].value_counts())

In [ ]:
# CELL 3: PREPARE FEATURES AND LABELS
# Get descriptor columns
descriptor_cols = [col for col in df_hopv.columns if col not in ['smiles', 'homo', 'lumo', 'stability']]

X_hopv = df_hopv[descriptor_cols].values
y_hopv = df_hopv['stability'].values

# Encode labels
le = LabelEncoder()
y_hopv = le.fit_transform(y_hopv)

print(f"Features: {X_hopv.shape}")
print(f"Encoded classes: {dict(zip(le.classes_, range(len(le.classes_))))}")    

In [ ]:
# CELL 4: COMPUTE KAGGLE DESCRIPTORS
descriptor_names = [x[0] for x in Descriptors._descList]

def compute_descriptors(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    return [getattr(Descriptors, name)(mol) for name in descriptor_names]

X_kaggle = pd.DataFrame(
    [compute_descriptors(s) for s in df_kaggle['smiles'].tolist()], 
    columns=descriptor_names
)

print(f"Kaggle features: {X_kaggle.shape}")

In [ ]:
# CELL 5: TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_hopv, y_hopv, test_size=0.2, random_state=42, stratify=y_hopv
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"\nTrain class distribution:")
for cls, count in zip(le.classes_, np.bincount(y_train)):
    print(f"  {cls}: {count}")
print(f"\nTest class distribution:")
for cls, count in zip(le.classes_, np.bincount(y_test)):
    print(f"  {cls}: {count}")

In [ ]:
# CELL 6: PREPROCESSING
# Fit preprocessing on train data only
var_sel = VarianceThreshold(threshold=0)
imp = SimpleImputer(strategy='median')

# Transform train
X_train_clean = var_sel.fit_transform(X_train)
X_train_imp = imp.fit_transform(X_train_clean)

# Transform test
X_test_clean = var_sel.transform(X_test)
X_test_imp = imp.transform(X_test_clean)

# Transform Kaggle
X_kaggle_clean = var_sel.transform(X_kaggle)
X_kaggle_imp = imp.transform(X_kaggle_clean)

print(f"Features after preprocessing: {X_train_imp.shape[1]}")

In [ ]:
    # CELL 7: FEATURE SELECTION
# Best k value (from your 0.90 model)
k_features = 40

sel = SelectKBest(f_classif, k=k_features)
X_train_sel = sel.fit_transform(X_train_imp, y_train)
X_test_sel = sel.transform(X_test_imp)
X_kaggle_sel = sel.transform(X_kaggle_imp)

print(f"Selected {k_features} features")
print(f"Train shape: {X_train_sel.shape}")
print(f"Test shape: {X_test_sel.shape}")
print(f"Kaggle shape: {X_kaggle_sel.shape}")

In [ ]:
# CELL 8: SCALING
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled = scaler.transform(X_test_sel)
X_kaggle_scaled = scaler.transform(X_kaggle_sel)

print(f"✓ Data scaled")
print(f"Train: {X_train_scaled.shape}")
print(f"Test: {X_test_scaled.shape}")
print(f"Kaggle: {X_kaggle_scaled.shape}")

In [ ]:
# CELL 9: BUILD AND TRAIN MODEL
print("="*60)
print("BUILDING CLASSIFICATION MODEL")
print("="*60)

# Best parameters from your 0.90 model
# You mentioned it was LogisticRegression trained on train set
# The reference had: CV=0.896, Test=0.9533, predictions H=106, L=30, M=64


# Replace your LogisticRegression with:
lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
)
# Cross-validation on train set
cv_scores = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring='f1_macro')

print(f"\nCV F1 scores per fold:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\nCV F1 mean: {cv_scores.mean():.4f}")
print(f"CV F1 std:  {cv_scores.std():.4f}")

In [ ]:
# CELL 10: TRAIN ON FULL TRAIN SET AND EVALUATE ON TEST
# Train on training set
lr.fit(X_train_scaled, y_train)

# Evaluate on test set
test_pred = lr.predict(X_test_scaled)
test_f1 = f1_score(y_test, test_pred, average='macro')

print("="*60)
print("TEST SET PERFORMANCE")
print("="*60)
print(f"Test F1 Score: {test_f1:.4f}")

print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print(f"\nClassification Report:")
print(classification_report(y_test, test_pred, target_names=le.classes_))

print("✓ Model evaluation complete")

In [ ]:
# CELL 11: PREDICT ON KAGGLE TEST SET
pred_kaggle = lr.predict(X_kaggle_scaled)
pred_kaggle_labels = le.inverse_transform(pred_kaggle)

pred_dist = dict(zip(*np.unique(pred_kaggle_labels, return_counts=True)))

print("="*60)
print("KAGGLE PREDICTIONS")
print("="*60)
print(f"Prediction distribution:")
print(f"  high:   {pred_dist.get('high', 0)}")
print(f"  low:    {pred_dist.get('low', 0)}")
print(f"  medium: {pred_dist.get('medium', 0)}")

print(f"\nReference (0.90 Kaggle score):")
print(f"  high:   101")
print(f"  low:    24")
print(f"  medium: 75")

print(f"\n✓ Predictions complete")

In [ ]:
# CELL 12: CREATE AND SAVE SUBMISSION
submission = pd.DataFrame({
    'smiles': df_kaggle['smiles'],
    'stability': pred_kaggle_labels
})

submission.to_csv('../../../data/003_kaggle_competition_data/25261_classification_sample_submission.csv', index=False)

print("="*60)
print("SUBMISSION CREATED")
print("="*60)
print(f"File: submission_classification_hopv.csv")
print(f"Rows: {len(submission)}")
print(f"\nFirst 10 predictions:")
print(submission.head(10))
print("\n✓✓✓ Ready to submit! ✓✓✓")
print("="*60)

In [ ]:
# CELL 15: VISUALIZATION SETUP
import matplotlib.pyplot as plt
import os

# Create figures directory if it doesn't exist
os.makedirs('../../../figures', exist_ok=True)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("✓ Visualization setup ready")
print(f"✓ Figures directory: {os.path.abspath('../../../figures')}")

In [ ]:
# CELL 13: SUMMARY
print("="*60)
print("FINAL MODEL SUMMARY")
print("="*60)
print(f"Dataset: HOPV only ({len(df_hopv)} samples)")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Features: {k_features} selected by f_classif")
print(f"\nPerformance:")
print(f"  - CV F1:   {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
print(f"  - Test F1: {test_f1:.4f}")
print(f"\nKaggle Predictions:")
print(f"  - high:   {pred_dist.get('high', 0)}")
print(f"  - low:    {pred_dist.get('low', 0)}")
print(f"  - medium: {pred_dist.get('medium', 0)}")
print("="*60)

In [ ]:
# CELL 14: FEATURE IMPORTANCE ANALYSIS
print("="*60)
print("TOP SELECTED FEATURES")
print("="*60)

# Get the mask of selected features
selected_mask = sel.get_support()

# Get feature scores from SelectKBest
feature_scores = sel.scores_

# Map back to original feature names (after variance threshold)
# First get the features that passed variance threshold
var_mask = var_sel.get_support()
features_after_var = [descriptor_cols[i] for i in range(len(descriptor_cols)) if var_mask[i]]

# Now get selected features with their scores
selected_features = []
for i, (selected, score) in enumerate(zip(selected_mask, feature_scores)):
    if selected:
        selected_features.append({
            'feature': features_after_var[i],
            'f_score': score
        })

# Sort by score
selected_features = sorted(selected_features, key=lambda x: x['f_score'], reverse=True)

print(f"\nTop 10 features by F-score:")
for i, feat in enumerate(selected_features[:10], 1):
    print(f"  {i:2d}. {feat['feature']:30s} (F={feat['f_score']:.2f})")

print(f"\nAll {k_features} selected features:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i:2d}. {feat['feature']}")

In [ ]:
# CELL 15: VISUALIZATION SETUP
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("✓ Visualization setup ready")

# CELL 16: CLASS DISTRIBUTION - HOPV vs SQL COMPARISON
# Load SQL data for comparison (adjust path as needed)
try:
    df_sql = pd.read_csv('../../../data/002_processed_data/xy_df_sql.csv')
    df_sql = df_sql.dropna(subset=['stability'])
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # HOPV distribution
    hopv_counts = df_hopv['stability'].value_counts()
    class_order = ['high', 'low', 'medium']
    hopv_vals = [hopv_counts.get(c, 0) for c in class_order]
    colors = ['#2ecc71', '#e74c3c', '#f39c12']
    
    axes[0].bar(class_order, hopv_vals, color=colors, edgecolor='black', linewidth=1.2)
    axes[0].set_title('HOPV Dataset (Used for Training)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Stability Class')
    axes[0].set_ylabel('Number of Molecules')
    for i, val in enumerate(hopv_vals):
        axes[0].text(i, val + 2, str(val), ha='center', fontsize=12, fontweight='bold')
    
    # SQL distribution
    sql_counts = df_sql['stability'].value_counts()
    sql_vals = [sql_counts.get(c, 0) for c in class_order]
    
    axes[1].bar(class_order, sql_vals, color=colors, edgecolor='black', linewidth=1.2)
    axes[1].set_title('SQL Dataset (Excluded)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Stability Class')
    axes[1].set_ylabel('Number of Molecules')
    for i, val in enumerate(sql_vals):
        axes[1].text(i, val + 2, str(val), ha='center', fontsize=12, fontweight='bold')
    
    plt.suptitle('Class Distribution Comparison: Why HOPV-Only Performed Better', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('../../../figures/class_distribution_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Saved: class_distribution_comparison.png")
    
except FileNotFoundError:
    print("SQL dataset not found - creating HOPV-only visualization")
    fig, ax = plt.subplots(figsize=(8, 5))
    hopv_counts = df_hopv['stability'].value_counts()
    class_order = ['high', 'low', 'medium']
    hopv_vals = [hopv_counts.get(c, 0) for c in class_order]
    colors = ['#2ecc71', '#e74c3c', '#f39c12']
    
    ax.bar(class_order, hopv_vals, color=colors, edgecolor='black', linewidth=1.2)
    ax.set_title('HOPV Dataset Class Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Stability Class')
    ax.set_ylabel('Number of Molecules')
    for i, val in enumerate(hopv_vals):
        ax.text(i, val + 2, str(val), ha='center', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../../../figures/class_distribution_hopv.png', dpi=150, bbox_inches='tight')
    plt.show()

# CELL 17: CONFUSION MATRIX HEATMAP
fig, ax = plt.subplots(figsize=(8, 6))

cm = confusion_matrix(y_test, test_pred)
class_names = le.classes_

im = ax.imshow(cm, cmap='Blues')
cbar = ax.figure.colorbar(im, ax=ax, shrink=0.8)

ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)

thresh = cm.max() / 2
for i in range(len(class_names)):
    for j in range(len(class_names)):
        color = 'white' if cm[i, j] > thresh else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=14, fontweight='bold')

ax.set_title(f'Confusion Matrix (Test Set)\nF1 Score: {test_f1:.4f}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../../figures/confusion_matrix_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: confusion_matrix_classification.png")

# CELL 18: TOP FEATURES BAR CHART
fig, ax = plt.subplots(figsize=(10, 6))

top_n = 10
top_feats = selected_features[:top_n]
feat_names = [f['feature'] for f in top_feats]
feat_scores = [f['f_score'] for f in top_feats]

bars = ax.barh(range(len(feat_names)), feat_scores, color='#3498db', edgecolor='black', linewidth=0.8)
ax.set_yticks(range(len(feat_names)))
ax.set_yticklabels(feat_names)
ax.invert_yaxis()
ax.set_xlabel('F-Score (ANOVA)', fontsize=12)
ax.set_title('Top 10 Features Selected by SelectKBest (k=40)', fontsize=14, fontweight='bold')

for i, (bar, score) in enumerate(zip(bars, feat_scores)):
    ax.text(score + 2, i, f'{score:.1f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../../../figures/top_features_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: top_features_classification.png")

# CELL 19: KAGGLE PREDICTION DISTRIBUTION
fig, ax = plt.subplots(figsize=(8, 5))

classes = ['high', 'low', 'medium']
counts = [pred_dist.get(c, 0) for c in classes]
colors = ['#2ecc71', '#e74c3c', '#f39c12']

bars = ax.bar(classes, counts, color=colors, edgecolor='black', linewidth=1.2)

ax.set_title('Predicted Stability Distribution (Kaggle Test Set)', fontsize=14, fontweight='bold')
ax.set_xlabel('Stability Class')
ax.set_ylabel('Number of Molecules')

for bar, count in zip(bars, counts):
    pct = count / sum(counts) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
            f'{count}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../../../figures/kaggle_predictions_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: kaggle_predictions_distribution.png")

# CELL 20: CROSS-VALIDATION SCORES PLOT
fig, ax = plt.subplots(figsize=(8, 5))

folds = range(1, len(cv_scores) + 1)
ax.bar(folds, cv_scores, color='#9b59b6', edgecolor='black', linewidth=1.2, alpha=0.8)
ax.axhline(y=cv_scores.mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean: {cv_scores.mean():.4f}')
ax.fill_between([0.5, len(cv_scores) + 0.5], 
                cv_scores.mean() - cv_scores.std(), 
                cv_scores.mean() + cv_scores.std(), 
                alpha=0.2, color='#e74c3c', label=f'±1 Std: {cv_scores.std():.4f}')

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('F1 Score (Macro)', fontsize=12)
ax.set_title('5-Fold Cross-Validation Performance', fontsize=14, fontweight='bold')
ax.set_xticks(folds)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')

for fold, score in zip(folds, cv_scores):
    ax.text(fold, score + 0.02, f'{score:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../../../figures/cv_scores_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: cv_scores_classification.png")

# CELL 21: SUMMARY OF ALL FIGURES
print("="*60)
print("VISUALIZATION SUMMARY")
print("="*60)
print("\nGenerated figures:")
print("  1. class_distribution_comparison.png - HOPV vs SQL class balance")
print("  2. confusion_matrix_classification.png - Model performance")
print("  3. top_features_classification.png - Feature importance")
print("  4. kaggle_predictions_distribution.png - Final predictions")
print("  5. cv_scores_classification.png - Cross-validation stability")
print("\nAll figures saved to: ../../../figures/")
print("="*60)